# 🔒 Production Best Practices

**W tym notebooku:**
- Security - CORS, rate limiting, SQL injection, XSS, CSRF, SSL
- Performance - caching, database optimization, CDN
- Scaling - horizontal/vertical, load balancing, auto-scaling
- Database migrations i backups
- API versioning
- Infrastructure as Code
- Cost optimization

---

## 1. Security - OWASP Top 10

### OWASP Top 10 (najczęstsze zagrożenia):

```
1. Broken Access Control
2. Cryptographic Failures
3. Injection (SQL, XSS, etc.)
4. Insecure Design
5. Security Misconfiguration
6. Vulnerable Components
7. Authentication Failures
8. Software/Data Integrity Failures
9. Security Logging Failures
10. Server-Side Request Forgery (SSRF)
```

---

## 2. SQL Injection Prevention

### ❌ Vulnerable code:

```python
# NIGDY TAK NIE RÓB!
@app.get("/users")
async def get_users(username: str):
    # SQL Injection vulnerability!
    query = f"SELECT * FROM users WHERE username = '{username}'"
    result = await db.execute(query)
    return result

# Atak:
# GET /users?username=admin' OR '1'='1
# Query: SELECT * FROM users WHERE username = 'admin' OR '1'='1'
# Zwraca WSZYSTKICH użytkowników!

# Gorszy atak:
# GET /users?username=admin'; DROP TABLE users; --
# Usuwa całą tabelę!
```

### ✅ Bezpieczny kod (ORM):

```python
from sqlalchemy import select
from sqlalchemy.orm import Session

@app.get("/users")
async def get_users(username: str, db: Session = Depends(get_db)):
    # ✅ ORM automatycznie escape'uje
    query = select(User).where(User.username == username)
    result = await db.execute(query)
    return result.scalars().all()

# Atak nie zadziała:
# GET /users?username=admin' OR '1'='1
# ORM zamieni to na:
# SELECT * FROM users WHERE username = 'admin\' OR \'1\'=\'1'
# (escaped, treated as literal string)
```

### ✅ Bezpieczny kod (raw SQL z parametrami):

```python
# Jeśli MUSISZ użyć raw SQL:
@app.get("/users")
async def get_users(username: str):
    # ✅ Parametrized query
    query = "SELECT * FROM users WHERE username = :username"
    result = await db.execute(query, {"username": username})
    return result.fetchall()

# Database driver automatycznie escape'uje parametry
```

### Zasada:

```
✅ ZAWSZE używaj:
- ORM (SQLAlchemy, Django ORM)
- Parametrized queries (:param, $1, ?)

❌ NIGDY:
- String concatenation (f"{query}")
- String formatting (f"SELECT * FROM {table}")
```

---

## 3. XSS (Cross-Site Scripting) Prevention

### Problem:

```python
# User input:
comment = "<script>alert('XSS')</script>"

# ❌ Vulnerable (wyświetlenie bez escape):
# HTML: <div>{comment}</div>
# Browser wykonuje JavaScript!
```

### Rozwiązanie (FastAPI):

```python
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from markupsafe import escape

@app.post("/comments")
async def create_comment(comment: str):
    # ✅ Escape HTML
    safe_comment = escape(comment)
    # "<script>alert('XSS')</script>" →
    # "&lt;script&gt;alert('XSS')&lt;/script&gt;"
    
    await db.save_comment(safe_comment)
    return {"comment": safe_comment}
```

### FastAPI automatyczne escape:

```python
# FastAPI JSON responses są bezpieczne domyślnie
@app.get("/comments")
async def get_comments():
    comments = await db.get_comments()
    # JSON automatycznie escape'uje <, >, etc.
    return {"comments": comments}
```

### Content Security Policy (CSP):

```python
from fastapi.middleware.trustedhost import TrustedHostMiddleware
from fastapi import Response

@app.middleware("http")
async def add_security_headers(request, call_next):
    response = await call_next(request)
    
    # CSP header (blokuje inline scripts)
    response.headers["Content-Security-Policy"] = (
        "default-src 'self'; "
        "script-src 'self' https://trusted-cdn.com; "
        "style-src 'self' 'unsafe-inline';"
    )
    
    # Inne security headers
    response.headers["X-Content-Type-Options"] = "nosniff"
    response.headers["X-Frame-Options"] = "DENY"
    response.headers["X-XSS-Protection"] = "1; mode=block"
    
    return response
```

---

## 4. CORS (Cross-Origin Resource Sharing)

### Problem:

```
Frontend (https://myapp.com)
  ↓ API call
Backend (https://api.myapp.com)

Browser: "Different origin! Blocked by CORS policy"
```

### ❌ Insecure CORS (allow all):

```python
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # ❌ WSZYSTKIE domeny!
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Zagrożenie:
# Złośliwa strona (https://evil.com) może:
# - Wywołać API w imieniu użytkownika
# - Wykraść dane (jeśli allow_credentials=True)
```

### ✅ Secure CORS:

```python
from fastapi.middleware.cors import CORSMiddleware

# Production
ALLOWED_ORIGINS = [
    "https://myapp.com",
    "https://www.myapp.com",
]

# Development (tylko local)
if settings.ENVIRONMENT == "development":
    ALLOWED_ORIGINS.append("http://localhost:3000")

app.add_middleware(
    CORSMiddleware,
    allow_origins=ALLOWED_ORIGINS,  # ✅ Tylko zaufane domeny
    allow_credentials=True,
    allow_methods=["GET", "POST", "PUT", "DELETE"],  # Tylko potrzebne
    allow_headers=["Authorization", "Content-Type"],  # Tylko potrzebne
)
```

### CORS pre-flight request:

```
Browser:
  1. OPTIONS /api/users (pre-flight)
     Headers: Origin: https://myapp.com
  
Server:
  2. 200 OK
     Headers:
       Access-Control-Allow-Origin: https://myapp.com
       Access-Control-Allow-Methods: GET, POST

Browser:
  3. OK, teraz wysyłam prawdziwy request
     POST /api/users
```

---

## 5. Rate Limiting

### Po co rate limiting?

```
Bez rate limiting:
❌ Brute force attacks (1000 login attempts/sec)
❌ DDoS (miliony requestów)
❌ API abuse (scraping)
❌ Wysoki koszt infrastruktury
```

### FastAPI + SlowAPI:

```python
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded

limiter = Limiter(key_func=get_remote_address)
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

# Global limit
@app.get("/api/users")
@limiter.limit("100/minute")  # 100 requests per minute per IP
async def get_users():
    return await db.get_users()

# Stricter limit for expensive operations
@app.post("/api/search")
@limiter.limit("10/minute")
async def search(query: str):
    return await expensive_search(query)

# Very strict for authentication
@app.post("/api/login")
@limiter.limit("5/minute")  # Tylko 5 prób logowania/min
async def login(username: str, password: str):
    return await authenticate(username, password)
```

### nginx rate limiting:

```nginx
# nginx.conf
http {
    # Define rate limit zone
    limit_req_zone $binary_remote_addr zone=api_limit:10m rate=10r/s;
    
    server {
        location /api/ {
            # Apply rate limit
            limit_req zone=api_limit burst=20 nodelay;
            
            # rate=10r/s = 10 requests per second
            # burst=20 = allow burst up to 20 requests
            # nodelay = reject excess immediately (no queue)
            
            proxy_pass http://backend;
        }
    }
}
```

### Redis-based rate limiting (distributed):

```python
import redis
from fastapi import HTTPException, status

redis_client = redis.Redis(host='redis', port=6379)

async def rate_limit(key: str, limit: int, window: int):
    """
    key: unique identifier (user_id, IP)
    limit: max requests
    window: time window (seconds)
    """
    current = redis_client.incr(key)
    
    if current == 1:
        # First request - set expiry
        redis_client.expire(key, window)
    
    if current > limit:
        raise HTTPException(
            status_code=status.HTTP_429_TOO_MANY_REQUESTS,
            detail="Rate limit exceeded"
        )

@app.get("/api/users")
async def get_users(request: Request):
    # Rate limit: 100 requests per 60 seconds per IP
    await rate_limit(
        key=f"ratelimit:{request.client.host}",
        limit=100,
        window=60
    )
    
    return await db.get_users()
```

---

## 6. SSL/TLS (HTTPS)

### HTTP vs HTTPS:

```
HTTP (port 80):
❌ Plain text (widoczne dla każdego w sieci)
❌ Man-in-the-middle attacks
❌ No authentication (nie wiesz czy to prawdziwy serwer)

HTTPS (port 443):
✅ Encrypted (TLS)
✅ Authentication (SSL certificate)
✅ Data integrity (nie można zmodyfikować w tranzycie)
```

### Let's Encrypt (darmowy SSL):

```bash
# 1. Install Certbot
sudo apt-get install certbot python3-certbot-nginx

# 2. Obtain certificate
sudo certbot --nginx -d example.com -d www.example.com

# Certbot automatycznie:
# - Tworzy SSL certificate
# - Konfiguruje nginx (HTTPS)
# - Setupuje auto-renewal (certyfikat ważny 90 dni)

# 3. Test auto-renewal
sudo certbot renew --dry-run
```

### nginx SSL config (po Certbot):

```nginx
# nginx.conf (automatycznie dodane przez Certbot)
server {
    listen 443 ssl http2;
    server_name example.com;
    
    # SSL certificates
    ssl_certificate /etc/letsencrypt/live/example.com/fullchain.pem;
    ssl_certificate_key /etc/letsencrypt/live/example.com/privkey.pem;
    
    # SSL configuration
    ssl_protocols TLSv1.2 TLSv1.3;
    ssl_ciphers HIGH:!aNULL:!MD5;
    ssl_prefer_server_ciphers on;
    
    # HSTS (force HTTPS)
    add_header Strict-Transport-Security "max-age=31536000; includeSubDomains" always;
    
    location / {
        proxy_pass http://backend;
    }
}

# HTTP → HTTPS redirect
server {
    listen 80;
    server_name example.com;
    
    return 301 https://$server_name$request_uri;
}
```

### Docker + Let's Encrypt:

```yaml
# docker-compose.yml
services:
  nginx:
    image: nginx:alpine
    ports:
      - "80:80"
      - "443:443"
    volumes:
      - ./nginx/nginx.conf:/etc/nginx/nginx.conf
      - ./certbot/conf:/etc/letsencrypt  # SSL certificates
      - ./certbot/www:/var/www/certbot   # ACME challenge
  
  certbot:
    image: certbot/certbot
    volumes:
      - ./certbot/conf:/etc/letsencrypt
      - ./certbot/www:/var/www/certbot
    entrypoint: "/bin/sh -c 'trap exit TERM; while :; do certbot renew; sleep 12h & wait $${!}; done;'"
```

---

## 7. Caching - Performance Boost

### Typy cache:

```
1. Application Cache (Redis)
   ├─ Cache database queries
   └─ Cache expensive computations

2. HTTP Cache (nginx, CDN)
   ├─ Cache static files
   └─ Cache API responses

3. Database Query Cache
   └─ Built-in (PostgreSQL, MySQL)

4. Browser Cache
   └─ Cache-Control headers
```

### Redis caching (FastAPI):

```python
import redis
import json
from functools import wraps

redis_client = redis.Redis(host='redis', port=6379, decode_responses=True)

def cache(expire: int = 300):
    """Cache decorator (expire in seconds)"""
    def decorator(func):
        @wraps(func)
        async def wrapper(*args, **kwargs):
            # Generate cache key
            cache_key = f"{func.__name__}:{args}:{kwargs}"
            
            # Try to get from cache
            cached = redis_client.get(cache_key)
            if cached:
                return json.loads(cached)
            
            # Cache miss - execute function
            result = await func(*args, **kwargs)
            
            # Save to cache
            redis_client.setex(
                cache_key,
                expire,
                json.dumps(result)
            )
            
            return result
        return wrapper
    return decorator

@app.get("/users")
@cache(expire=600)  # Cache 10 minut
async def get_users():
    # Expensive database query
    users = await db.query(User).all()
    return users

# Pierwszy request: database query (slow)
# Kolejne requesty (10 min): Redis cache (fast!)
```

### Cache invalidation:

```python
@app.post("/users")
async def create_user(user: UserCreate):
    # Create user
    new_user = await db.create_user(user)
    
    # Invalidate cache (users list changed!)
    redis_client.delete("get_users:():{}")
    
    return new_user
```

### nginx caching:

```nginx
# nginx.conf
http {
    # Cache path
    proxy_cache_path /var/cache/nginx levels=1:2 keys_zone=api_cache:10m max_size=1g;
    
    server {
        location /api/public/ {
            proxy_pass http://backend;
            
            # Enable cache
            proxy_cache api_cache;
            proxy_cache_valid 200 10m;  # Cache 200 OK for 10 minutes
            proxy_cache_valid 404 1m;   # Cache 404 for 1 minute
            
            # Cache key
            proxy_cache_key "$scheme$request_method$host$request_uri";
            
            # Add cache status header
            add_header X-Cache-Status $upstream_cache_status;
        }
    }
}
```

### CDN (Content Delivery Network):

```
User (Poland)
  ↓
CDN Edge Server (Warsaw) - cached static files
  ↓ (cache miss)
Origin Server (US) - original files

Benefits:
✅ Faster (geographic proximity)
✅ Reduced origin load
✅ DDoS protection

CDN providers:
- Cloudflare (free tier!)
- AWS CloudFront
- Fastly
- Akamai
```

---

## 8. Database Optimization

### Indexes:

```python
# Without index:
# SELECT * FROM users WHERE email = 'test@example.com'
# → Full table scan (O(n)) - slow!

# With index:
from sqlalchemy import Index

class User(Base):
    __tablename__ = "users"
    
    id = Column(Integer, primary_key=True)
    email = Column(String, unique=True, index=True)  # ✅ Index!
    username = Column(String, index=True)  # ✅ Index!
    created_at = Column(DateTime, index=True)  # ✅ Index for sorting
    
    # Composite index (multiple columns)
    __table_args__ = (
        Index('ix_user_email_active', 'email', 'is_active'),
    )

# SELECT * FROM users WHERE email = 'test@example.com'
# → Index lookup (O(log n)) - fast!
```

### Connection Pooling:

```python
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

# ❌ Bez pooling:
# Każdy request: connect → query → disconnect
# Overhead: 10-50ms per request!

# ✅ Z pooling:
engine = create_engine(
    DATABASE_URL,
    poolclass=QueuePool,
    pool_size=10,        # 10 connections w pool
    max_overflow=20,     # +20 connections jeśli pool pełny
    pool_pre_ping=True,  # Test connection przed użyciem
    pool_recycle=3600,   # Recycle connection co 1h
)

# Request reuses existing connection from pool (fast!)
```

### N+1 Query Problem:

```python
# ❌ N+1 problem:
@app.get("/users-with-posts")
async def get_users_with_posts():
    users = await db.query(User).all()  # 1 query
    
    result = []
    for user in users:  # N queries (one per user!)
        posts = await db.query(Post).filter(Post.user_id == user.id).all()
        result.append({"user": user, "posts": posts})
    
    return result

# Total: 1 + N queries (jeśli 100 users = 101 queries!)

# ✅ Solution: Eager loading (JOIN)
from sqlalchemy.orm import selectinload

@app.get("/users-with-posts")
async def get_users_with_posts():
    users = await db.query(User).options(
        selectinload(User.posts)  # JOIN in one query
    ).all()
    
    return users

# Total: 2 queries (1 for users, 1 for all posts)
```

### Query optimization:

```python
# ❌ Fetch all (memory issue!):
users = await db.query(User).all()  # 1M users = OOM!

# ✅ Pagination:
@app.get("/users")
async def get_users(skip: int = 0, limit: int = 100):
    users = await db.query(User).offset(skip).limit(limit).all()
    return users

# ✅ Select only needed columns:
# ❌
users = await db.query(User).all()  # Fetches ALL columns

# ✅
from sqlalchemy import select
users = await db.execute(
    select(User.id, User.username)  # Only needed columns
)
```

---

## 9. Scaling Strategies

### Vertical Scaling (scale up):

```
Przed:
Server: 2 CPU, 4GB RAM

Po:
Server: 8 CPU, 32GB RAM

✅ Prosty (tylko upgrade servera)
✅ Brak zmian w aplikacji
❌ Limit (największy dostępny server)
❌ Single point of failure
❌ Downtime podczas upgrade
❌ Expensive (exponential cost)
```

### Horizontal Scaling (scale out) - RECOMMENDED:

```
Przed:
1× Server (2 CPU, 4GB RAM)

Po:
10× Servers (2 CPU, 4GB RAM each)

✅ Unlimited scaling
✅ High availability (jeśli 1 pada, 9 działa)
✅ Zero downtime deployment
✅ Cost-effective (linear cost)
❌ Wymaga stateless aplikacji
❌ Load balancer needed
```

### Load Balancing:

```nginx
# nginx load balancer
upstream backend {
    # Load balancing algorithms:
    
    # 1. Round-robin (default)
    server web1:8000;
    server web2:8000;
    server web3:8000;
    # Request 1 → web1, Request 2 → web2, Request 3 → web3, ...
    
    # 2. Least connections
    # least_conn;
    # Wysyła do servera z najmniejszą liczbą aktywnych połączeń
    
    # 3. IP hash (sticky sessions)
    # ip_hash;
    # Ten sam user zawsze na tym samym serverze
    
    # 4. Weighted
    # server web1:8000 weight=3;  # 3x więcej requestów
    # server web2:8000 weight=1;
}

server {
    location / {
        proxy_pass http://backend;
    }
}
```

### Auto-scaling (Kubernetes):

```yaml
# Horizontal Pod Autoscaler
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: fastapi-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: fastapi
  minReplicas: 2    # Minimum 2 pods
  maxReplicas: 10   # Maximum 10 pods
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70  # Scale jeśli CPU > 70%

# Kubernetes automatycznie:
# - Scale up (więcej pods) gdy CPU > 70%
# - Scale down (mniej pods) gdy CPU < 70%
```

### Database Scaling:

```
1. Read Replicas:
   Master (write)
     ↓ replicate
   Replica 1 (read)
   Replica 2 (read)
   Replica 3 (read)
   
   # Write queries → Master
   # Read queries → Replicas (load balanced)

2. Sharding (horizontal partitioning):
   Shard 1: users (id 1-1M)
   Shard 2: users (id 1M-2M)
   Shard 3: users (id 2M-3M)
   
   # Distribute data across multiple databases

3. Connection Pooling:
   # Reuse connections (już omówione)
```

---

## 10. Database Migrations (Alembic)

### Po co migrations?

```
Problem bez migrations:
❌ Manual ALTER TABLE (error-prone)
❌ Brak version control schema
❌ Trudny rollback
❌ Różne schema na dev/staging/prod

Z migrations:
✅ Version control schema
✅ Automated schema changes
✅ Rollback capability
✅ Sync schema across environments
```

### Alembic setup:

```bash
# 1. Install
pip install alembic

# 2. Initialize
alembic init alembic

# Creates:
# alembic/
#   ├── env.py         # Migration environment
#   ├── script.py.mako # Migration template
#   └── versions/      # Migration files
# alembic.ini          # Alembic config
```

### Create migration:

```bash
# Auto-generate migration (detects model changes)
alembic revision --autogenerate -m "create users table"

# Creates: alembic/versions/abc123_create_users_table.py
```

```python
# alembic/versions/abc123_create_users_table.py
from alembic import op
import sqlalchemy as sa

# Revision ID
revision = 'abc123'
down_revision = None  # Previous migration (None = first)

def upgrade():
    """Apply migration"""
    op.create_table(
        'users',
        sa.Column('id', sa.Integer, primary_key=True),
        sa.Column('email', sa.String, nullable=False),
        sa.Column('username', sa.String, nullable=False),
        sa.Column('created_at', sa.DateTime, server_default=sa.func.now()),
    )
    op.create_index('ix_users_email', 'users', ['email'])

def downgrade():
    """Rollback migration"""
    op.drop_index('ix_users_email')
    op.drop_table('users')
```

### Apply migrations:

```bash
# Apply all pending migrations
alembic upgrade head

# Rollback one migration
alembic downgrade -1

# Rollback to specific revision
alembic downgrade abc123

# Show current revision
alembic current

# Show migration history
alembic history
```

### Production deployment with migrations:

```bash
# 1. Backup database (ALWAYS!)
pg_dump mydb > backup_$(date +%Y%m%d).sql

# 2. Run migrations BEFORE deploying new code
docker-compose run --rm web alembic upgrade head

# 3. Deploy new code
docker-compose up -d --build

# 4. Verify (rollback if issues)
curl https://api.example.com/health
```

### Zero-downtime migrations:

```python
# ❌ Breaking change (downtime!):
def upgrade():
    op.drop_column('users', 'old_field')  # Old code crashes!

# ✅ Zero-downtime strategy (multi-step):

# Step 1: Add new column (nullable)
def upgrade():
    op.add_column('users', sa.Column('new_field', sa.String, nullable=True))
# Deploy code that uses new_field (but handles null)

# Step 2: Backfill data
def upgrade():
    op.execute("UPDATE users SET new_field = old_field")

# Step 3: Make NOT NULL
def upgrade():
    op.alter_column('users', 'new_field', nullable=False)

# Step 4: Remove old column (after all instances use new_field)
def upgrade():
    op.drop_column('users', 'old_field')
```

---

## 11. Backups & Disaster Recovery

### Backup strategy (3-2-1 rule):

```
3 copies of data:
├─ 1. Production database (primary)
├─ 2. Local backup (same datacenter)
└─ 3. Remote backup (different datacenter/cloud)

2 different storage types:
├─ Disk
└─ Cloud (S3, Google Cloud Storage)

1 off-site backup:
└─ Different geographic location
```

### PostgreSQL backup:

```bash
# Full database backup
pg_dump -U user -h localhost dbname > backup.sql

# Compressed backup
pg_dump -U user dbname | gzip > backup.sql.gz

# Custom format (parallel restore, partial restore)
pg_dump -U user -Fc dbname > backup.dump

# Restore
psql -U user dbname < backup.sql
pg_restore -U user -d dbname backup.dump
```

### Automated backups (cron):

```bash
#!/bin/bash
# backup.sh

DATE=$(date +%Y%m%d_%H%M%S)
BACKUP_DIR="/backups"
DB_NAME="mydb"

# Create backup
pg_dump -U postgres $DB_NAME | gzip > "$BACKUP_DIR/backup_${DATE}.sql.gz"

# Upload to S3
aws s3 cp "$BACKUP_DIR/backup_${DATE}.sql.gz" s3://my-backups/

# Delete local backups older than 7 days
find $BACKUP_DIR -name "backup_*.sql.gz" -mtime +7 -delete

# Delete S3 backups older than 30 days
aws s3 ls s3://my-backups/ | awk '{print $4}' | while read file; do
    # ... delete old files ...
done
```

```bash
# Crontab (daily backup at 2 AM)
0 2 * * * /usr/local/bin/backup.sh
```

### Docker volume backup:

```bash
# Backup volume
docker run --rm \
  -v myapp_postgres_data:/data \
  -v $(pwd):/backup \
  alpine tar czf /backup/postgres_backup.tar.gz /data

# Restore volume
docker run --rm \
  -v myapp_postgres_data:/data \
  -v $(pwd):/backup \
  alpine tar xzf /backup/postgres_backup.tar.gz -C /
```

### Point-in-time recovery (PostgreSQL):

```sql
-- Enable WAL archiving (postgresql.conf)
wal_level = replica
archive_mode = on
archive_command = 'cp %p /archive/%f'

-- Recover to specific point in time
-- 1. Restore base backup
-- 2. Create recovery.conf:
restore_command = 'cp /archive/%f %p'
recovery_target_time = '2024-01-15 14:30:00'
```

### Test backups regularly!

```bash
# Monthly backup test
# 1. Restore backup to test database
# 2. Verify data integrity
# 3. Document restore time
```

---

## 12. API Versioning

### Po co versioning?

```
Problem bez versioning:
❌ Breaking changes → apps crashują
❌ Nie możesz zmienić response format
❌ Backward compatibility nightmare

Z versioning:
✅ Breaking changes w v2, v1 nadal działa
✅ Gradual migration (clients upgrade when ready)
✅ Deprecated versions (sunset timeline)
```

### URL versioning (RECOMMENDED):

```python
from fastapi import FastAPI, APIRouter

app = FastAPI()

# Version 1
router_v1 = APIRouter(prefix="/api/v1")

@router_v1.get("/users")
async def get_users_v1():
    return [{"id": 1, "name": "John"}]  # Old format

# Version 2 (breaking change)
router_v2 = APIRouter(prefix="/api/v2")

@router_v2.get("/users")
async def get_users_v2():
    return {
        "users": [{"id": 1, "full_name": "John Doe"}],  # New format!
        "total": 1
    }

app.include_router(router_v1)
app.include_router(router_v2)

# Clients:
# Old app: GET /api/v1/users (still works)
# New app: GET /api/v2/users (new format)
```

### Header versioning:

```python
from fastapi import Header

@app.get("/api/users")
async def get_users(api_version: str = Header(default="1")):
    if api_version == "2":
        return get_users_v2()
    else:
        return get_users_v1()

# Request:
# GET /api/users
# Headers: API-Version: 2
```

### Deprecation policy:

```python
from fastapi import Response
import warnings

@router_v1.get("/users")
async def get_users_v1(response: Response):
    # Mark as deprecated
    response.headers["Deprecation"] = "true"
    response.headers["Sunset"] = "2024-12-31"  # Will be removed
    response.headers["Link"] = '</api/v2/users>; rel="successor-version"'
    
    warnings.warn(
        "API v1 is deprecated. Please migrate to v2 by 2024-12-31",
        DeprecationWarning
    )
    
    return [{"id": 1, "name": "John"}]
```

---

## 13. Production Checklist

### Security ✅

```
✅ HTTPS/SSL enabled (Let's Encrypt)
✅ CORS configured (nie "*")
✅ Rate limiting (SlowAPI/nginx)
✅ SQL injection protection (ORM)
✅ XSS protection (escape output)
✅ Secrets w environment variables (nie hardcoded)
✅ Security headers (CSP, HSTS, X-Frame-Options)
✅ Dependencies scanning (Dependabot, Snyk)
```

### Performance ✅

```
✅ Caching (Redis)
✅ Database indexes
✅ Connection pooling
✅ CDN dla static files
✅ Gzip compression (nginx)
✅ Query optimization (brak N+1)
✅ Pagination (nie fetch all)
```

### Reliability ✅

```
✅ Health checks (/health, /readiness)
✅ Monitoring (Sentry, Prometheus)
✅ Logging (structured, JSON)
✅ Automated backups (daily, tested)
✅ CI/CD pipeline
✅ Zero-downtime deployment
✅ Rollback plan
```

### Scalability ✅

```
✅ Stateless aplikacja
✅ Horizontal scaling ready
✅ Load balancer (nginx/cloud)
✅ Database read replicas
✅ Auto-scaling (Kubernetes HPA)
```

### DevOps ✅

```
✅ Docker/docker-compose
✅ Infrastructure as Code (Terraform)
✅ Database migrations (Alembic)
✅ Secrets management (Vault/AWS Secrets)
✅ Documentation (README, API docs)
```

---

## 📚 Podsumowanie

### Co się nauczyłeś:

✅ **Security:**
- SQL injection prevention (ORM, parametrized queries)
- XSS protection (escape output, CSP)
- CORS configuration (whitelist origins)
- Rate limiting (SlowAPI, nginx, Redis)
- SSL/TLS (Let's Encrypt)

✅ **Performance:**
- Caching (Redis, nginx, CDN)
- Database optimization (indexes, pooling, N+1)
- Query optimization (pagination, select columns)

✅ **Scaling:**
- Horizontal vs Vertical scaling
- Load balancing (nginx, round-robin, least-conn)
- Auto-scaling (Kubernetes HPA)
- Database scaling (read replicas, sharding)

✅ **Operations:**
- Database migrations (Alembic)
- Backups & disaster recovery (3-2-1 rule)
- API versioning (URL, header)
- Production checklist

### Minimum Viable Production (MVP):

```
1. ✅ HTTPS (Let's Encrypt)
2. ✅ Basic security (CORS, rate limiting)
3. ✅ Health checks
4. ✅ Error tracking (Sentry)
5. ✅ Automated backups (daily)
6. ✅ CI/CD pipeline
7. ✅ Logging (structured)
```

### Enterprise Production:

```
MVP +
8. ✅ Full monitoring (Prometheus + Grafana)
9. ✅ Log aggregation (ELK/Loki)
10. ✅ Auto-scaling (Kubernetes)
11. ✅ Multi-region deployment
12. ✅ CDN (Cloudflare/CloudFront)
13. ✅ Database replicas
14. ✅ Advanced caching (Redis cluster)
15. ✅ Disaster recovery plan (tested!)
```

### Kluczowe wnioski:

```
Security = Ochrona PRZED problemem
  ↓
Performance = Szybkość dla użytkowników
  ↓
Scaling = Obsługa rosnącego traffic
  ↓
Operations = Utrzymanie i recovery
```

**Pamiętaj:**
- Security first (nie można dodać later)
- Test backups regularnie
- Monitor wszystko (nie zgaduj)
- Scale horizontally (nie vertically)
- Automate wszystko (CI/CD, backups, deployments)

---

## 🎓 Całkowite podsumowanie (Notebooki 01-08)

### Journey od development do production:

```
01. HTTP Server i WSGI
    ↓ (jak serwer rozmawia z Pythonem)
02. WSGI vs ASGI
    ↓ (sync vs async, kiedy co)
03. Uvicorn, Gunicorn
    ↓ (implementacje serwerów)
04. nginx i Reverse Proxy
    ↓ (warstwa przed aplikacją)
05. Production Stack
    ↓ (kompletna architektura)
06. Docker Deployment
    ↓ (konteneryzacja)
07. CI/CD & Monitoring
    ↓ (automatyzacja i obserwacja)
08. Best Practices
    ↓ (security, performance, scaling)
```

### Production-ready aplikacja = wszystkie 8 elementów!

**Powodzenia w produkcji! 🚀**